In [28]:
import geopandas as gpd
import os
import warnings

warnings.filterwarnings("ignore")

BUFFER_SIZES = [1.5, 3.0, 6.0]
GSD = 1.0  # Ground Sampling Distance


def compute_acc_with_gsd(gt_shp, pred_shp, buffer_size=0.5, gsd=1.0):
    gt_gdf = gpd.read_file(gt_shp)
    pred_gdf = gpd.read_file(pred_shp)

    gt_gdf = gt_gdf[gt_gdf.geometry.type == "LineString"]
    pred_gdf = pred_gdf[pred_gdf.geometry.type == "LineString"]

    pred_buffer = pred_gdf.geometry.buffer(buffer_size)

    total_gt_points = 0
    points_inside_buffer = 0

    for line in gt_gdf.geometry:
        line_length = line.length
        num_points = int(line_length / gsd)
        sampled_points = [line.interpolate(i * gsd) for i in range(num_points)]

        for pt in sampled_points:
            if any(pred_buffer.contains(pt)):
                points_inside_buffer += 1

        total_gt_points += len(sampled_points)

    acc = points_inside_buffer / total_gt_points if total_gt_points > 0 else 0
    return round(acc, 4)

def compute_metrics(gt_path, pred_path):
    print(f"\nMetrics:")
    for buffer in BUFFER_SIZES:
        acc = compute_acc_with_gsd(gt_path, pred_path, buffer_size=buffer, gsd=GSD)
        print(f"  Accuracy @{buffer}m buffer: {acc}")

In [29]:
compute_metrics(pred_path=r"C:\Users\User\Downloads\HLR_edge_shp(2)\lines.shp", gt_path=r"D:\Mines\CIL\Nigahi\out\test-nigahi\results\Edges.shp")


Metrics:
  Accuracy @1.5m buffer: 0.2303
  Accuracy @3.0m buffer: 0.4824
  Accuracy @6.0m buffer: 0.7439


In [30]:
import geopandas as gpd
from shapely.geometry import MultiPolygon

def calculate_iou(gt_medians, pred_medians):
    """
    Calculate Intersection over Union (IoU) between ground truth medians and predicted medians.
    This function merges all polygons in each GeoDataFrame into a MultiPolygon and computes the IoU.

    :param gt_medians: GeoDataFrame containing ground truth polygons
    :param pred_medians: GeoDataFrame containing predicted polygons
    :return: None (prints the IoU and the areas of intersection and union)
    """
    # Check if the CRS are the same, and reproject if necessary
    if gt_medians.crs != pred_medians.crs:
        print("CRS mismatch detected. Reprojecting predictions to the CRS of ground truth.")
        pred_medians = pred_medians.to_crs(gt_medians.crs)
    
    # Merge all ground truth polygons into a single MultiPolygon
    gt_multi_polygon = gt_medians.geometry.unary_union
    # Merge all predicted polygons into a single MultiPolygon
    pred_multi_polygon = pred_medians.geometry.unary_union
    
    # Calculate intersection and union of the two MultiPolygons
    intersection = gt_multi_polygon.intersection(pred_multi_polygon)
    union = gt_multi_polygon.union(pred_multi_polygon)

    print(gt_multi_polygon.area, pred_multi_polygon.area, intersection.area, union.area)
    
    # Calculate areas for intersection and union
    intersection_area = intersection.area if intersection.is_valid else 0
    union_area = union.area if union.is_valid else 0
    
    # Compute IoU
    if union_area > 0:
        iou = intersection_area / pred_multi_polygon.area
        print(f"Intersection Area: {intersection_area:.4f}")
        print(f"Union Area: {union_area:.4f}")
        print(f"IoU: {iou:.4f}")
    else:
        print("Union area is zero, cannot compute IoU.")

# Load the shapefiles into GeoDataFrames
gt_medians_gdf = gpd.read_file(r"C:\Users\User\Downloads\HLR_medians_shp(2)\polygons.shp")
pred_medians_gdf = gpd.read_file(r"D:\Mines\CIL\Nigahi\out\test-nigahi-3\results\Medians.shp")

# Now call the function with the loaded GeoDataFrames
calculate_iou(gt_medians_gdf, pred_medians_gdf)


66428.88517285736 5088.7082020234775 2709.9124207034365 68807.68095417906
Intersection Area: 2709.9124
Union Area: 68807.6810
IoU: 0.5325


In [26]:
gt_medians = r"C:\Users\User\Downloads\HLR_medians_shp\polygons.shp"
pred_medians = r"D:\Mines\CIL\Nigahi\out\test-nigahi\results\Medians.shp"

calculate_iou(gt_medians, pred_medians)

AttributeError: 'str' object has no attribute 'crs'